In [ ]:
!pip install langchain langchain-community langchain-huggingface
!pip install faiss-cpu sentence-transformers pypdf transformers accelerate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 52.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 34.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.9 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 55.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.4/331.4 kB 19.4 MB/s eta 0:00:00


In [ ]:
from google.colab import files
uploaded = files.upload()

Saving Annual-Report-FY-2023-24.pdf to Annual-Report-FY-2023-24.pdf


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = PyPDFLoader("Annual-Report-FY-2023-24.pdf")
documents = loader.load()

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=200
)

docs = splitter.split_documents(documents)

print("Total chunks:", len(docs))

Total chunks: 91


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(docs, embeddings)

retriever = vectorstore.as_retriever(search_kwargs={"k":3})

In [ ]:
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=120,
    do_sample=True,
    temperature=0.2,
    pad_token_id=tokenizer.eos_token_id
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [ ]:
def ask_question(query):

    docs = retriever.invoke(query)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
Answer the question using the context below.

Context:
{context}

Question: {query}

Answer in one short sentence:
"""

    result = generator(prompt)

    output = result[0]["generated_text"]

    answer = output.replace(prompt, "").strip()

    # stop at first line
    answer = answer.split("\n")[0]

    print("Final Answer:")
    print(answer)

    print("\nSupporting Context:")
    print(context[:500])

In [ ]:
ask_question("Who is the CEO of Swiggy?")

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Final Answer:
Sriharsha Majety is the CEO of Swiggy.

Supporting Context:
2. Names of associates or joint ventures which have been liquidated or sold during the year: NIL  
For and on Behalf of the Board of Directors of  
Swiggy Limited 
 
 
 
          Sd/-                                          Sd/-                                               Sd/-                                Sd/- 
 
Sriharsha Majety  
Managing Director & 
Group CEO 
(DIN: 06680073) 
 
Lakshmi Nandan Reddy Obul  
Whole time Director – Head of 
Innovation  
(DIN: 06686145) 
 
Rahul Bothra 
Chie


In [ ]:
ask_question("What services does Swiggy provide?")

Both `max_new_tokens` (=120) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Final Answer:
Swiggy offers a wide range of food and grocery delivery services, including food delivery, quick commerce, and a growing array of household items delivered to users in less than 10-15 minutes.

Supporting Context:
cost of the Company. Your directors are confident that the Company will grow and prosper in the coming 
years. 
 
2. RESULTS OF OPERATIONS AND THE STATE OF COMPANY’S AFFAIRS  
 
History: 
 
At Swiggy, our mission is to elevate the quality of life for urban consumers by offering unparalleled 
convenience. Innovation has been an integral part of our DNA which encourages us to ideate, experiment 
and iterate constantly with the focus on identifying and addressing convenience needs of our users at t
